# 삼성전자 4Q25 — Multi-Query RAG (OpenAI 버전)

**Try0-Multi-Query_RAG_OpenAI에서 다른점**

- PDFLoader의 변경사항
```
기존 PyPDF -> UnstructuredPDFLoader using Tesseract (Image detection)
```

## 1. 환경 설정 및 라이브러리 임포트

In [1]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI   # ← HuggingFace에서 변경
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.load import dumps, loads
from operator import itemgetter
from langchain_community.document_loaders import UnstructuredPDFLoader

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

# API 키 확인
if openai_api_key:
    print(f"✅ OpenAI API 키 로드 완료: sk-...{openai_api_key[-4:]}")
else:
    print("❌ API 키가 없습니다. .env 파일을 확인하세요.")

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ OpenAI API 키 로드 완료: sk-...yK4A


**If UnstructuredPDFLoader library is not downloaded:**

unstructured package not found, please install it with `pip install "unstructured[pdf]"`

pip install "unstructured[pdf]"을 하게 되면 hi_res의 추가 의존성 관련 패키지를 한번에 설치해줌
- pdfminer.six
- pikepdf
- pillow-heif

등 PDF 처리에 필요한 패키지들이 함께 설치됌

```
Tesseract 설치 및 Path 등록
URL: https://github.com/UB-Mannheim/tesseract/wiki
- 64비트 폴더 다운

components to install에서 additional language - korean 체크 확인*

다운로드 이후
1. 시스템 환경 변수 편집 --> 환경 변수 클릭
2. Path 선택: 편집
3. 새로 만들기: C:\Program Files\Tesseract-OCR 등록

설치 확인
1. 새 터미널(cmd) 에 tesseract --version
2. VSCode 터미널에도 확인: 만약 안뜨면 컴퓨터 재시작
```

In [2]:
Powerpoint_Document_path= "C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf"
PDF_Document_path = "C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf"

docs = []

# 페이지 단위가 아닌 "요소(element)" 단위로 문서를 분리함
loader1 = UnstructuredPDFLoader(
    Powerpoint_Document_path,
    # detectron2 또는 yolox 같은 컴피터 비전 모델 활용하여 차트 확인 // 테세락트 설치 필요
    strategy="hi_res",
    # Tells loader NOT to save cropped images of every picture in finds in my PDF to local drive
    # Since most LLMs are text-based, you only want the description or data from the image, not the image itself
    extract_images_in_pdf=False,
    # when hi_res finds a table, this setting will force the loader to reconstruct that table into a strcutured format, and will read using infer_table_structure
    infer_table_structure=True,
    languages = ["eng"] # 한국어 파일이라면 "kor"
)

loader2 = PyPDFLoader(PDF_Document_path)
docs.extend(loader1.load())
docs.extend(loader2.load())

print(f"총 로드된 페이지 수: {len(docs)}")

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


총 로드된 페이지 수: 35


In [3]:
# 확인용 — 어느 파일에서 왔는지 출력
for doc in docs[:5]:
    print(doc.metadata.get("source"), "|", doc.metadata.get("page_number"))

print(docs[0])
print(docs[1])

C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf | None
C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf | None
page_content='SAMSUNG ELECTRONICS

Earnings Presentation: 4Q 2025 Financial Results

SAMSUNG

Disclaimer

The financial information in this document are consolidated earnings results based on K-IFRS.

This document is provided for the convenience of investors only before the external audit on our 4Q 2025 financial results is completed. The Audit outcomes may cause some parts of this document to change.

This document contains "forward-looking statements" - that is statements related to future not past events. In this context "forward-looking statements" often address our expected future business and financial performance and often contain words su

## 3. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,    
    chunk_overlap=220
)

splits = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(splits)}\n")
print(f"첫번째 청크 예시:\n{splits[0].page_content}\n")
print(f"두번째 청크 예시:\n{splits[1].page_content}")

분할된 청크 수: 75

첫번째 청크 예시:
SAMSUNG ELECTRONICS

Earnings Presentation: 4Q 2025 Financial Results

SAMSUNG

Disclaimer

The financial information in this document are consolidated earnings results based on K-IFRS.

This document is provided for the convenience of investors only before the external audit on our 4Q 2025 financial results is completed. The Audit outcomes may cause some parts of this document to change.

This document contains "forward-looking statements" - that is statements related to future not past events. In this context "forward-looking statements" often address our expected future business and financial performance and often contain words such as "expects" "anticipates" "intends" "plans" "believes" "seeks" or "will". "Forward-looking statements" by their nature address matters that are to different degrees uncertain. For us particular uncertainties which could adversely or positively affect our future results include:

-

-

-

The behavior of financial markets includi

## 4. Vector Store

**변경점:**
- 임베딩: `BAAI/bge-m3` → `text-embedding-3-small`
- 벡터스토어: `FAISS` → `Chroma`
- GPU 불필요 (API 호출 방식)

**에러 발생시**
https://developers.openai.com/api/docs/guides/error-codes#api-errors 을 통해 에러 확인: 높은 확률로 Billing set-up 안되어 있기 때문

429 에러: 빌링 이슈

https://platform.openai.com/settings/organization/billing/overview 에서 빌링 업데이트 필수

https://platform.openai.com/settings/organization/limits 리밋 확대 URL


In [5]:
# OpenAI 임베딩 (API 호출 방식 — GPU 불필요)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Chroma 벡터스토어
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

# Multi-Query: 쿼리당 k=4 (5쿼리 × 4 = 최대 20개 → 중복 제거)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Vector store 준비 완료")

Vector store 준비 완료


## 5. LLM 설정

**변경점:** `gemma-2-9b-it` → `gpt-4o-mini`

gpt-4o-mini를 선택한 이유:
- 수치 추론 및 테이블 해석 능력이 Gemma-9b 대비 훨씬 뛰어남
- 불확실한 컨텍스트에서 hallucination 대신 "모르겠다"고 답하는 경향
- 비용: 질문 1번당 약 $0.0005 (약 0.7원)

In [6]:
chat_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=openai_api_key
)

print("LLM 준비 완료: gpt-4o-mini")

LLM 준비 완료: gpt-4o-mini


## 6. Multi-Query RAG 구성

---여기서부터는 HuggingFace 버전과 완전히 동일---

In [7]:
# Step 1: Multi-Query 생성 체인
# 사용자 질문 1개 → LLM이 5가지 다른 관점으로 재작성
QUERY_PROMPT = ChatPromptTemplate.from_template("""
You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. 

Original question: {question}
""")

query_chain = (
    QUERY_PROMPT
    | chat_llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

print("Multi-Query 체인 준비 완료")

Multi-Query 체인 준비 완료


In [8]:
# Step 2: Unique Union
# 5개의 검색 결과를 합치고 중복 제거
def get_unique_union(documents: list[list]):
    """
    5개 쿼리로 검색된 문서 리스트를 받아 중복을 제거하고 반환.
    
    Args:
        documents: [[doc1, doc2], [doc2, doc3], ...] 형태
    Returns:
        중복이 제거된 unique Document 리스트
    """
    # 중첩 리스트 flatten + Document → JSON string 변환 (set 비교용)
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # 중복 제거
    unique_docs = list(set(flattened_docs))
    # 다시 Document 객체로 복원
    return [loads(doc) for doc in unique_docs]

In [9]:
# retrieval_chain: 질문 → 5개 재작성 → 병렬 검색 → 중복 제거
retrieval_chain = query_chain | retriever.map() | get_unique_union

## 7. 최종 RAG 체인 구성

In [10]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an expert AI assistant specializing in Samsung Electronics earnings reports.
Answer the question based ONLY on the provided context.
If the context does not contain enough information to answer confidently,
say "The provided context does not contain sufficient information on this topic."
Always include specific numbers and financial figures when available.

#Context:
{context}

#Question:
{question}

#Answer:
""")

final_rag_chain = (
    {
        "context": retrieval_chain,
        "question": itemgetter("question")
    }
    | RAG_PROMPT
    | chat_llm
    | StrOutputParser()
)

print("최종 RAG 체인 준비 완료")

최종 RAG 체인 준비 완료


## 8. 실행 및 테스트

In [11]:
from langchain_teddynote.messages import stream_response

In [ ]:
# 테스트 1: 전반적인 실적
question = "삼성전자 2025 4분기 실적 발표에 대해서 알려줘"
answer = final_rag_chain.invoke({"question": question})

stream_response(answer)
# Langsmith Trace
# https://smith.langchain.com/public/6f8575d7-8d16-460a-b9b8-76b9ed61bcda/r

C:\Users\user\AppData\Local\Temp\ipykernel_5028\3162270408.py:17: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


삼성전자의 2025년 4분기 실적 발표에 따르면, 매출은 86.1조 원으로 전년 동기 대비 증가하였으며, 운영 이익은 20.1조 원으로 증가했습니다. 운영 마진은 21.4%로, 이전 분기 대비 7.3%포인트 상승했습니다. DX 부문에서는 MX 및 가전 사업의 둔화로 운영 이익이 감소했지만, DS 부문은 메모리 수익성의 강력한 개선 덕분에 분기 대비 성장을 이뤘습니다. 또한, 미국 달러와 기타 주요 통화의 급격한 상승이 회사 전체 운영 이익에 약 1.6조 원의 긍정적인 영향을 미쳤습니다. 2025년 전체 연간 매출은 역사상 최고치를 기록했습니다.